# LLM free features + DL tuning — Claude API free features

Colab-ready notebook. It reuses saved feature CSV files and never calls an LLM/provider API.

In [ ]:
import importlib.util, subprocess, sys, os

IN_COLAB = "google.colab" in sys.modules
INSTALL_TABPFN = os.getenv("INSTALL_TABPFN", "1") == "1"
RUN_TABPFN_DEFAULT = os.getenv("RUN_TABPFN", "1") == "1"
TABPFN_ENABLE_HF_LOGIN = True
TABPFN_HF_TOKEN = os.getenv("TABPFN_TOKEN", "") or os.getenv("HF_TOKEN", "") or os.getenv("HUGGING_FACE_HUB_TOKEN", "")

REQUIRED_PACKAGES = ["optuna"]
TABPFN_HELPER_PACKAGES = ["huggingface_hub"]

def ensure_package(module_name: str, pip_name: str | None = None):
    pip_name = pip_name or module_name
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
    else:
        print(f"{module_name} already installed")

for pkg in REQUIRED_PACKAGES:
    ensure_package(pkg)

TABPFN_HF_LOGIN_STATUS = "not_requested"
if INSTALL_TABPFN:
    for pkg in TABPFN_HELPER_PACKAGES:
        ensure_package(pkg)
    ensure_package("tabpfn")
    if TABPFN_ENABLE_HF_LOGIN:
        try:
            if not TABPFN_HF_TOKEN and IN_COLAB:
                from google.colab import userdata
                for key in ["TABPFN_TOKEN", "HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HUGGINGFACE_TOKEN"]:
                    try:
                        TABPFN_HF_TOKEN = userdata.get(key) or ""
                    except Exception:
                        TABPFN_HF_TOKEN = ""
                    if TABPFN_HF_TOKEN:
                        break
            from huggingface_hub import login
            if TABPFN_HF_TOKEN:
                login(token=str(TABPFN_HF_TOKEN).strip(), add_to_git_credential=False, skip_if_logged_in=True)
                TABPFN_HF_LOGIN_STATUS = "ok"
                print("HF login for TabPFN: OK")
            else:
                TABPFN_HF_LOGIN_STATUS = "missing_token"
                print("HF login for TabPFN: token not provided. Continuing without explicit login.")
        except Exception as e:
            TABPFN_HF_LOGIN_STATUS = f"failed:{type(e).__name__}"
            print(f"HF login for TabPFN failed: {type(e).__name__}: {e}")
else:
    print("TabPFN installation disabled")


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import json
import time
import shutil
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error
from sklearn.metrics import pairwise_distances

SEED = 2
seed = SEED
np.random.seed(SEED)
random.seed(SEED)

target_col = "duration_hours"
cap = 960
IN_COLAB = "google.colab" in sys.modules
USE_GOOGLE_DRIVE = True
MOUNT_GOOGLE_DRIVE = True

PROVIDER = "claude"
PROVIDER_TITLE = "Claude API free features"
BACKEND_TAG = "claude_api_claude_sonnet_4_20250514"
DRIVE_PROJECT_REL = "llm_local_colab_runs/claude_api_free_features"
LOCAL_PROJECT_REL = "llm_local_runs/claude_api_free_features"
ARTIFACT_DIR_NAME = "artifacts_claude_api_free_features"

if IN_COLAB and USE_GOOGLE_DRIVE and MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE_RUNTIME_DIR = Path("/content/drive/MyDrive")
else:
    BASE_RUNTIME_DIR = Path(".")

LOCAL_PROJECT_DIR = Path(LOCAL_PROJECT_REL)
PROVIDER_RUN_DIR_NAME = Path(DRIVE_PROJECT_REL).name

PROJECT_DIR_CANDIDATES = [
    BASE_RUNTIME_DIR / DRIVE_PROJECT_REL,
    BASE_RUNTIME_DIR / LOCAL_PROJECT_REL,
    BASE_RUNTIME_DIR / "llm_local_runs" / PROVIDER_RUN_DIR_NAME,
    BASE_RUNTIME_DIR / "llm_local_colab_runs" / PROVIDER_RUN_DIR_NAME,
    Path(LOCAL_PROJECT_REL),
    Path("llm_local_runs") / PROVIDER_RUN_DIR_NAME,
    Path("drive_artifacts") / PROVIDER_RUN_DIR_NAME,
]

ARTIFACT_DIR_CANDIDATES = []
for _project_dir in PROJECT_DIR_CANDIDATES:
    ARTIFACT_DIR_CANDIDATES.extend([
        _project_dir / ARTIFACT_DIR_NAME,
        _project_dir / "artifacts" / ARTIFACT_DIR_NAME,
    ])
if IN_COLAB and USE_GOOGLE_DRIVE:
    ARTIFACT_DIR_CANDIDATES.extend([
        BASE_RUNTIME_DIR / "drive_artifacts" / PROVIDER_RUN_DIR_NAME / ARTIFACT_DIR_NAME,
        BASE_RUNTIME_DIR / ARTIFACT_DIR_NAME,
    ])

ARTIFACT_DIR = next((p for p in ARTIFACT_DIR_CANDIDATES if p.exists()), None)
if ARTIFACT_DIR is None:
    # Default expected location. Feature loading below prints all checked candidates if files are missing.
    ARTIFACT_DIR = BASE_RUNTIME_DIR / DRIVE_PROJECT_REL / ARTIFACT_DIR_NAME

DRIVE_PROJECT_DIR = ARTIFACT_DIR.parent
INPUT_DIR = DRIVE_PROJECT_DIR / "inputs"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
DL_ARTIFACT_DIR = ARTIFACT_DIR / "dl_tuning"
DL_BASELINE_ARTIFACT_DIR = ARTIFACT_DIR / "dl_baseline_default_params"
DL_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
DL_BASELINE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

DATA_CANDIDATES = [
    INPUT_DIR / "data_finall_without_TTM.csv",
    DRIVE_PROJECT_DIR / "data_finall_without_TTM.csv",
    Path("./data_finall_without_TTM.csv"),
    Path("./data_finall.csv"),
    Path("/content/data_finall_without_TTM.csv"),
    Path("/content/data_finall.csv"),
    Path("/mnt/data/data_finall_without_TTM.csv"),
    Path("/mnt/data/data_finall.csv"),
]

FINAL_TUNED_REFERENCE_CANDIDATES = [
    INPUT_DIR / "final_summary.csv",
    DRIVE_PROJECT_DIR / "final_summary.csv",
    Path("./final_summary.csv"),
    Path("/content/final_summary.csv"),
    Path("/mnt/data/final_summary.csv"),
]

print("Provider:", PROVIDER_TITLE)
print("Project dir:", DRIVE_PROJECT_DIR.resolve())
print("Artifact dir:", ARTIFACT_DIR.resolve())
print("DL artifact dir:", DL_ARTIFACT_DIR.resolve())


In [ ]:
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Dataset not found. Put data_finall_without_TTM.csv next to the notebook, "
        "or in the provider Drive project/inputs folder."
    )

df = pd.read_csv(DATA_PATH)
if target_col not in df.columns:
    raise ValueError(f"Missing target column: {target_col}")

split = int(len(df) * 0.8)
train_full = df.iloc[:split].copy()
test_full = df.iloc[split:].copy()
test_typical = test_full[test_full[target_col] < cap].copy()
train_filt = train_full[train_full[target_col] < cap].copy()

meta_Xtr0 = train_filt.drop(columns=[target_col], errors="ignore").reset_index(drop=True)
meta_ytr0 = train_filt[target_col].reset_index(drop=True)
meta_Xte0 = test_full.drop(columns=[target_col], errors="ignore").reset_index(drop=True)
meta_yte0 = test_full[target_col].reset_index(drop=True)
meta_Xte_typ0 = test_typical.drop(columns=[target_col], errors="ignore").reset_index(drop=True)
meta_yte_typ0 = test_typical[target_col].reset_index(drop=True)

test_typical_mask = (test_full[target_col].reset_index(drop=True) < cap)

bool_cols = meta_Xtr0.select_dtypes(include=["bool"]).columns.tolist()
for _frame in (meta_Xtr0, meta_Xte0, meta_Xte_typ0):
    if bool_cols:
        _frame[bool_cols] = _frame[bool_cols].astype(np.uint8)

non_numeric = meta_Xtr0.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    raise ValueError("Expected numeric prepared data, found non-numeric columns: " + ", ".join(non_numeric[:10]))

print("DATA_PATH     :", DATA_PATH)
print("train_full    :", train_full.shape)
print("train_typical :", train_filt.shape)
print("test_full     :", test_full.shape)
print("test_typical  :", test_typical.shape)
print("meta_Xtr0     :", meta_Xtr0.shape)
print("meta_Xte0     :", meta_Xte0.shape, "| full holdout")
print("meta_Xte_typ0 :", meta_Xte_typ0.shape, "| primary holdout")


In [ ]:

def make_clusterer(name, params):
    if name == "MiniBatchKMeans":
        return MiniBatchKMeans(
            n_clusters=params["n_clusters"],
            random_state=seed,
            n_init=20,
            batch_size=1024
        )
    elif name == "GaussianMixture":
        return GaussianMixture(
            n_components=params["n_components"],
            covariance_type=params["covariance_type"],
            random_state=seed
        )
    else:
        raise ValueError(name)

raw_cluster_scaler = StandardScaler()
raw_Xtr_scaled = raw_cluster_scaler.fit_transform(meta_Xtr0)
raw_Xte_scaled = raw_cluster_scaler.transform(meta_Xte0)

raw_cluster_pca = PCA(n_components=min(30, meta_Xtr0.shape[1]), random_state=seed)
raw_Xtr_embed = raw_cluster_pca.fit_transform(raw_Xtr_scaled)
raw_Xte_embed = raw_cluster_pca.transform(raw_Xte_scaled)

def probs_from_dist(all_d):
    inv = 1.0 / (all_d + 1e-6)
    return inv / inv.sum(axis=1, keepdims=True)

def fit_clusterer_and_assign(name, params, Xtr, Xte):
    est = make_clusterer(name, params)

    if name == "MiniBatchKMeans":
        est.fit(Xtr)
        tr_labels = est.predict(Xtr).astype(int)
        te_labels = est.predict(Xte).astype(int)

        if len(np.unique(tr_labels)) < 2:
            return None

        tr_d = pairwise_distances(Xtr, est.cluster_centers_)
        te_d = pairwise_distances(Xte, est.cluster_centers_)
        tr_proba = probs_from_dist(tr_d)
        te_proba = probs_from_dist(te_d)
        return tr_labels, te_labels, tr_proba, te_proba, "native"

    if name == "GaussianMixture":
        est.fit(Xtr)
        tr_proba = est.predict_proba(Xtr)
        te_proba = est.predict_proba(Xte)

        tr_labels = np.argmax(tr_proba, axis=1).astype(int)
        te_labels = np.argmax(te_proba, axis=1).astype(int)

        if len(np.unique(tr_labels)) < 2:
            return None

        return tr_labels, te_labels, tr_proba, te_proba, "native"

    return None

def build_cluster_meta_features(tr_labels, te_labels, tr_proba=None, te_proba=None):
    tr_feat = pd.DataFrame({"cluster_id": tr_labels})
    te_feat = pd.DataFrame({"cluster_id": te_labels})

    tr_sizes = pd.Series(tr_labels).value_counts().to_dict()
    tr_feat["cluster_size_train"] = pd.Series(tr_labels).map(tr_sizes).fillna(0)
    te_feat["cluster_size_train"] = pd.Series(te_labels).map(tr_sizes).fillna(0)

    tr_feat["is_noise"] = (tr_feat["cluster_id"] == -1).astype(int)
    te_feat["is_noise"] = (te_feat["cluster_id"] == -1).astype(int)

    tr_ohe = pd.get_dummies(tr_feat["cluster_id"], prefix="cluster")
    te_ohe = pd.get_dummies(te_feat["cluster_id"], prefix="cluster").reindex(columns=tr_ohe.columns, fill_value=0)

    tr_feat = pd.concat([tr_feat, tr_ohe], axis=1)
    te_feat = pd.concat([te_feat, te_ohe], axis=1)

    if tr_proba is not None and te_proba is not None:
        for k in range(tr_proba.shape[1]):
            tr_feat[f"cluster_proba_{k}"] = tr_proba[:, k]
            te_feat[f"cluster_proba_{k}"] = te_proba[:, k]

    return tr_feat, te_feat

FIXED_CLUSTER_CONFIGS = [
    {
        "tag": "gmm_diag_5",
        "clusterer": "GaussianMixture",
        "params": {"n_components": 5, "covariance_type": "diag"},
    },
    {
        "tag": "mbkm_2",
        "clusterer": "MiniBatchKMeans",
        "params": {"n_clusters": 2},
    },
]

raw_cluster_variants = {}
raw_cluster_summary_rows = []

for cfg in FIXED_CLUSTER_CONFIGS:
    print("Trying raw clusterer:", cfg)
    result = fit_clusterer_and_assign(cfg["clusterer"], cfg["params"], raw_Xtr_embed, raw_Xte_embed)
    if result is None:
        raise RuntimeError(f"Cluster config failed: {cfg}")

    tr_labels, te_labels, tr_proba, te_proba, fit_mode = result
    tr_feat, te_feat = build_cluster_meta_features(tr_labels, te_labels, tr_proba, te_proba)

    raw_cluster_variants[cfg["tag"]] = {
        "cfg": cfg,
        "fit_mode": fit_mode,
        "tr_labels": tr_labels,
        "te_labels": te_labels,
        "tr_proba": tr_proba,
        "te_proba": te_proba,
        "train_feat_raw": tr_feat,
        "test_feat_raw": te_feat,
    }

    raw_cluster_summary_rows.append({
        "tag": cfg["tag"],
        "clusterer": cfg["clusterer"],
        "params": json.dumps(cfg["params"], ensure_ascii=False),
        "fit_mode": fit_mode,
        "n_clusters_train": int(len(np.unique(tr_labels))),
        "train_rows": int(len(tr_labels)),
        "test_rows": int(len(te_labels)),
    })

raw_cluster_summary_df = pd.DataFrame(raw_cluster_summary_rows)
display(raw_cluster_summary_df)


In [ ]:
def feature_file_candidates(filename):
    candidates = [
        ARTIFACT_DIR / filename,
        INPUT_DIR / filename,
        DRIVE_PROJECT_DIR / filename,
        LOCAL_PROJECT_DIR / ARTIFACT_DIR_NAME / filename,
        Path("./") / filename,
        Path("/content") / filename,
        Path("/mnt/data") / filename,
    ]
    candidates.extend([p / filename for p in ARTIFACT_DIR_CANDIDATES])
    # preserve order while removing duplicates
    seen = set()
    out = []
    for p in candidates:
        key = str(p)
        if key not in seen:
            seen.add(key)
            out.append(p)
    return out

def find_feature_file(filename):
    for p in feature_file_candidates(filename):
        try:
            if p.exists():
                return p
        except Exception:
            pass
    raise FileNotFoundError(
        f"Saved feature CSV not found: {filename}\nChecked:\n" +
        "\n".join(str(p) for p in feature_file_candidates(filename))
    )

def load_saved_feature_frame(split_name, variant_name, expected_rows):
    filename = f"llm_features_{BACKEND_TAG}_{variant_name}_{split_name}.csv"
    path = find_feature_file(filename)
    feat = pd.read_csv(path)
    if "row_id" in feat.columns:
        feat = feat.sort_values("row_id").drop(columns=["row_id"]).reset_index(drop=True)
    else:
        feat = feat.reset_index(drop=True)
    if len(feat) != expected_rows:
        raise ValueError(f"{filename}: expected {expected_rows} rows, got {len(feat)}")
    for c in feat.columns:
        feat[c] = pd.to_numeric(feat[c], errors="coerce")
    if "parse_error" not in feat.columns:
        feat["parse_error"] = 0
    numeric_cols = [c for c in feat.columns if c != "parse_error"]
    if numeric_cols:
        medians = feat[numeric_cols].median(numeric_only=True)
        feat[numeric_cols] = feat[numeric_cols].fillna(medians).fillna(0)
    feat["parse_error"] = feat["parse_error"].fillna(0).astype(int)
    print(f"Loaded {variant_name}:{split_name} -> {path} | shape={feat.shape} | parse_errors={int(feat['parse_error'].sum())}")
    return feat

def concat_numeric(*frames):
    out = pd.concat([f.reset_index(drop=True) for f in frames], axis=1)
    out = out.loc[:, ~out.columns.duplicated()].copy()
    for c in out.columns:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    out = out.fillna(out.median(numeric_only=True)).fillna(0)
    return out

# 1) llm_only: meta + saved provider free features
llm_train_feat_only = load_saved_feature_frame("train", "llm_only", len(meta_Xtr0))
llm_test_feat_only = load_saved_feature_frame("test", "llm_only", len(meta_Xte0))

Xtr_llm_only = concat_numeric(meta_Xtr0, llm_train_feat_only)
Xte_llm_only_full = concat_numeric(meta_Xte0, llm_test_feat_only)
Xte_llm_only_typ = Xte_llm_only_full.loc[test_typical_mask.values].reset_index(drop=True)

print("LLM only train/full/typ:", Xtr_llm_only.shape, Xte_llm_only_full.shape, Xte_llm_only_typ.shape)


In [ ]:
cluster_before_llm_variants = {}

for tag, bundle in raw_cluster_variants.items():
    variant_name = f"cluster_before_llm__{tag}"
    llm_train_feat_cluster_before = load_saved_feature_frame("train", variant_name, len(meta_Xtr0))
    llm_test_feat_cluster_before = load_saved_feature_frame("test", variant_name, len(meta_Xte0))

    Xtr_cluster_before_llm = concat_numeric(meta_Xtr0, bundle["train_feat_raw"], llm_train_feat_cluster_before)
    Xte_cluster_before_llm_full = concat_numeric(meta_Xte0, bundle["test_feat_raw"], llm_test_feat_cluster_before)
    Xte_cluster_before_llm_typ = Xte_cluster_before_llm_full.loc[test_typical_mask.values].reset_index(drop=True)

    cluster_before_llm_variants[tag] = {
        "Xtr": Xtr_cluster_before_llm,
        "Xte_full": Xte_cluster_before_llm_full,
        "Xte_typ": Xte_cluster_before_llm_typ,
        "llm_train_feat": llm_train_feat_cluster_before,
        "llm_test_feat": llm_test_feat_cluster_before,
        "cfg": bundle["cfg"],
    }

    print("Variant:", variant_name)
    print("train/full/typ:", Xtr_cluster_before_llm.shape, Xte_cluster_before_llm_full.shape, Xte_cluster_before_llm_typ.shape)


In [ ]:
Xtr_llm_space = Xtr_llm_only.copy()
Xte_llm_space = Xte_llm_only_full.copy()

cluster_after_scaler = StandardScaler()
Xtr_llm_scaled = cluster_after_scaler.fit_transform(Xtr_llm_space)
Xte_llm_scaled = cluster_after_scaler.transform(Xte_llm_space)

cluster_after_pca = PCA(n_components=min(30, Xtr_llm_space.shape[1]), random_state=seed)
Xtr_llm_embed = cluster_after_pca.fit_transform(Xtr_llm_scaled)
Xte_llm_embed = cluster_after_pca.transform(Xte_llm_scaled)

llm_then_cluster_variants = {}
cluster_after_summary_rows = []

for cfg in FIXED_CLUSTER_CONFIGS:
    print("\n" + "=" * 95)
    print("Trying cluster-after-LLM:", cfg)
    print("=" * 95)

    result = fit_clusterer_and_assign(cfg["clusterer"], cfg["params"], Xtr_llm_embed, Xte_llm_embed)
    if result is None:
        raise RuntimeError(f"Cluster-after-LLM config failed: {cfg}")

    aft_tr_labels, aft_te_labels, aft_tr_proba, aft_te_proba, aft_fit_mode = result
    cluster_train_feat_after_llm, cluster_test_feat_after_llm = build_cluster_meta_features(
        aft_tr_labels, aft_te_labels, aft_tr_proba, aft_te_proba
    )

    Xtr_llm_then_cluster = concat_numeric(meta_Xtr0, llm_train_feat_only, cluster_train_feat_after_llm)
    Xte_llm_then_cluster_full = concat_numeric(meta_Xte0, llm_test_feat_only, cluster_test_feat_after_llm)
    Xte_llm_then_cluster_typ = Xte_llm_then_cluster_full.loc[test_typical_mask.values].reset_index(drop=True)

    tag = cfg["tag"]
    llm_then_cluster_variants[tag] = {
        "Xtr": Xtr_llm_then_cluster,
        "Xte_full": Xte_llm_then_cluster_full,
        "Xte_typ": Xte_llm_then_cluster_typ,
        "cfg": cfg,
        "fit_mode": aft_fit_mode,
    }

    cluster_after_summary_rows.append({
        "tag": tag,
        "clusterer": cfg["clusterer"],
        "params": json.dumps(cfg["params"], ensure_ascii=False),
        "fit_mode": aft_fit_mode,
        "n_clusters_train": int(len(np.unique(aft_tr_labels))),
    })

    print("Variant:", f"llm_then_cluster__{tag}")
    print("train/full/typ:", Xtr_llm_then_cluster.shape, Xte_llm_then_cluster_full.shape, Xte_llm_then_cluster_typ.shape)

cluster_after_summary_df = pd.DataFrame(cluster_after_summary_rows)
display(cluster_after_summary_df)


## DL tuning (24 core architectures + GrowNet + optional TabPFN)

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import time, math, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

try:
    from tabpfn import TabPFNRegressor
    HAS_TABPFN = True
except ImportError:
    HAS_TABPFN = False

SEED = 2
seed = SEED
torch.manual_seed(seed)
np.random.seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print('Device:', device, '| TabPFN:', HAS_TABPFN)

def seed_everything(s=SEED):
    import random
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

INPUT_DIM = 1

# ═══════════════════════════════════════════════════════════════
#  2. АРХИТЕКТУРЫ  (24 шт.)
# ═══════════════════════════════════════════════════════════════

# ── 2.1  MLP ─────────────────────────────────────────────────

class MLP(nn.Module):
    def __init__(self, d_in, hidden_dims, dropout=0.3):
        super().__init__()
        layers = []
        prev = d_in
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h),
                       nn.SiLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)


# ── 2.2  ResMLP ──────────────────────────────────────────────

class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim),
        )
        self.act = nn.SiLU()
    def forward(self, x):
        return self.act(x + self.block(x))

class ResMLP(nn.Module):
    def __init__(self, d_in, hidden=256, n_blocks=3, dropout=0.3):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU())
        self.blocks = nn.Sequential(*[ResBlock(hidden, dropout) for _ in range(n_blocks)])
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        return self.head(self.blocks(self.proj(x)))


# ── 2.3  SNN ─────────────────────────────────────────────────

class SNN(nn.Module):
    def __init__(self, d_in, hidden_dims=[256, 128], alpha_dropout=0.1):
        super().__init__()
        layers = []
        prev = d_in
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.SELU(), nn.AlphaDropout(alpha_dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='linear')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    def forward(self, x):
        return self.net(x)


# ── 2.4  GatedMLP ───────────────────────────────────────────

class GatedBlock(nn.Module):
    def __init__(self, d_in, d_out, dropout):
        super().__init__()
        self.linear = nn.Linear(d_in, d_out * 2)
        self.bn = nn.BatchNorm1d(d_out)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        h = self.linear(x)
        h1, h2 = h.chunk(2, dim=-1)
        return self.drop(self.bn(h1 * torch.sigmoid(h2)))

class GatedMLP(nn.Module):
    def __init__(self, d_in, hidden=256, n_blocks=3, dropout=0.3):
        super().__init__()
        blocks = [GatedBlock(d_in, hidden, dropout)]
        for _ in range(n_blocks - 1):
            blocks.append(GatedBlock(hidden, hidden, dropout))
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        return self.head(self.blocks(x))


# ── 2.5  GANDALF ─────────────────────────────────────────────

class GANDALF(nn.Module):
    def __init__(self, d_in, hidden=256, n_blocks=3, dropout=0.3):
        super().__init__()
        self.gate_fc = nn.Linear(d_in, d_in)
        layers = [nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout)]
        for _ in range(n_blocks - 1):
            layers += [nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout)]
        self.dnn = nn.Sequential(*layers)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        gate = torch.sigmoid(self.gate_fc(x))
        return self.head(self.dnn(x * gate))


# ── 2.6  DAE-MLP ────────────────────────────────────────────

class DAEMLP(nn.Module):
    def __init__(self, d_in, hidden=256, latent=64, noise=0.3, dropout=0.3):
        super().__init__()
        self.noise = noise
        self.encoder = nn.Sequential(
            nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, latent), nn.BatchNorm1d(latent), nn.SiLU(),
        )
        self.decoder = nn.Sequential(nn.Linear(latent, hidden), nn.SiLU(), nn.Linear(hidden, d_in))
        self.reg_head = nn.Sequential(nn.Dropout(dropout), nn.Linear(latent, 32), nn.SiLU(), nn.Linear(32, 1))
        self.aux_loss = torch.tensor(0.0)
    def forward(self, x):
        x_in = x * (torch.rand_like(x) > self.noise).float() if self.training and self.noise > 0 else x
        z = self.encoder(x_in)
        if self.training:
            self.aux_loss = F.mse_loss(self.decoder(z), x)
        return self.reg_head(z)


# ── 2.7  CNN1D ───────────────────────────────────────────────

class CNN1D(nn.Module):
    def __init__(self, d_in, channels=[64, 128, 64], ks=5, dropout=0.3):
        super().__init__()
        layers = []
        in_ch = 1
        for ch in channels:
            layers += [nn.Conv1d(in_ch, ch, ks, padding=ks // 2),
                       nn.BatchNorm1d(ch), nn.SiLU(), nn.Dropout(dropout)]
            in_ch = ch
        self.conv = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(nn.Linear(channels[-1], 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.pool(self.conv(x))
        return self.head(x.squeeze(-1))


# ── 2.8  BiGRU ───────────────────────────────────────────────

class BiGRU(nn.Module):
    def __init__(self, d_in, hidden=64, n_layers=2, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(input_size=1, hidden_size=hidden, num_layers=n_layers,
                          batch_first=True, bidirectional=True,
                          dropout=dropout if n_layers > 1 else 0)
        self.head = nn.Sequential(nn.Linear(hidden * 2, 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        _, h = self.gru(x.unsqueeze(-1))
        return self.head(torch.cat([h[-2], h[-1]], dim=1))


# ── 2.9  FT-Transformer ─────────────────────────────────────

class FTTransformer(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos = nn.Parameter(torch.randn(1, d_in + 1, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
              dim_feedforward=d_model * 4, dropout=dropout, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        B = x.shape[0]
        tok = self.feat_emb(x.unsqueeze(-1))
        tok = torch.cat([self.cls.expand(B, -1, -1), tok], dim=1) + self.pos
        return self.head(self.transformer(tok)[:, 0])


# ── 2.10  TabTransformer ────────────────────────────────────

class TabTransformer(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, mlp_hidden=128, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.col_emb = nn.Parameter(torch.randn(1, d_in, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
              dim_feedforward=d_model * 4, dropout=dropout, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.Linear(d_in * d_model + d_in, mlp_hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1))
    def forward(self, x):
        tok = self.feat_emb(x.unsqueeze(-1)) + self.col_emb
        out = self.transformer(tok)
        pooled = out.reshape(out.size(0), -1)
        return self.head(torch.cat([pooled, x], dim=1))


# ── 2.11  SAINT ──────────────────────────────────────────────

class SAINTBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.sa_norm = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ia_norm = nn.LayerNorm(d_model)
        self.inter_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model * 4),
                                 nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model * 4, d_model), nn.Dropout(dropout))
    def forward(self, x):
        h = self.sa_norm(x)
        h, _ = self.self_attn(h, h, h)
        x = x + h
        B, F_, D = x.shape
        if B > 1 and self.training:
            xt = x.permute(1, 0, 2)
            h2 = self.ia_norm(xt)
            h2, _ = self.inter_attn(h2, h2, h2)
            x = (xt + h2).permute(1, 0, 2)
        return x + self.ffn(x)

class SAINT(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos = nn.Parameter(torch.randn(1, d_in + 1, d_model) * 0.02)
        self.blocks = nn.ModuleList([SAINTBlock(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        B = x.shape[0]
        tok = self.feat_emb(x.unsqueeze(-1))
        tok = torch.cat([self.cls.expand(B, -1, -1), tok], dim=1) + self.pos
        for blk in self.blocks:
            tok = blk(tok)
        return self.head(tok[:, 0])


# ── 2.12  AutoInt ────────────────────────────────────────────

class AutoIntLayer(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.W_res = nn.Linear(d_model, d_model, bias=False)
    def forward(self, x):
        h, _ = self.attn(x, x, x)
        return F.relu(self.norm(h + self.W_res(x)))

class AutoInt(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=3, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.layers = nn.ModuleList([AutoIntLayer(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.Linear(d_in * d_model, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 1))
    def forward(self, x):
        tok = self.feat_emb(x.unsqueeze(-1))
        for layer in self.layers:
            tok = layer(tok)
        return self.head(tok.reshape(tok.size(0), -1))


# ── 2.13  Trompt ─────────────────────────────────────────────

class TromptLayer(nn.Module):
    def __init__(self, d_in, d_model, n_heads, dropout):
        super().__init__()
        self.prompt = nn.Parameter(torch.randn(1, d_in, d_model) * 0.02)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_model * 2), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(d_model * 2, d_model))
        self.norm2 = nn.LayerNorm(d_model)
        self.gate = nn.Linear(d_model, 1)
    def forward(self, feat_emb, x_raw):
        B = feat_emb.shape[0]
        prompted = feat_emb + self.prompt.expand(B, -1, -1)
        h, _ = self.attn(prompted, prompted, prompted)
        h = self.norm1(prompted + h)
        h = self.norm2(h + self.ffn(h))
        weights = torch.softmax(self.gate(h).squeeze(-1), dim=1)
        return h, weights * x_raw

class Trompt(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=3, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.layers = nn.ModuleList([TromptLayer(d_in, d_model, n_heads, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.Linear(d_in, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, 1))
    def forward(self, x):
        tok = self.feat_emb(x.unsqueeze(-1))
        preds = []
        for layer in self.layers:
            tok, lp = layer(tok, x)
            preds.append(lp)
        return self.head(torch.stack(preds).mean(0))


# ── 2.14  ExcelFormer ───────────────────────────────────────

class ExcelFormer(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.importance = nn.Parameter(torch.zeros(d_in))
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos = nn.Parameter(torch.randn(1, d_in + 1, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
              dim_feedforward=d_model * 4, dropout=dropout, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        B = x.shape[0]
        mask = torch.sigmoid(self.importance)
        x_masked = x * mask
        tok = self.feat_emb(x_masked.unsqueeze(-1))
        tok = torch.cat([self.cls.expand(B, -1, -1), tok], dim=1) + self.pos
        return self.head(self.transformer(tok)[:, 0])


# ── 2.15  ARM-Net ────────────────────────────────────────────

class ARMNet(nn.Module):
    def __init__(self, d_in, n_exp=64, hidden=128, order=2, dropout=0.2):
        super().__init__()
        self.order = order
        self.exp_layers = nn.ModuleList([nn.Linear(d_in, n_exp) for _ in range(order)])
        self.gate = nn.Sequential(nn.Linear(d_in, n_exp * order), nn.Sigmoid())
        self.head = nn.Sequential(
            nn.BatchNorm1d(n_exp * order), nn.Linear(n_exp * order, hidden),
            nn.SiLU(), nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, x):
        interactions = []
        for i, layer in enumerate(self.exp_layers):
            h = layer(x)
            if i > 0:
                h = h * interactions[-1]
            interactions.append(F.softplus(h))
        cat = torch.cat(interactions, dim=1)
        return self.head(cat * self.gate(x))


# ── 2.16  NODE ───────────────────────────────────────────────

class NODE(nn.Module):
    def __init__(self, d_in, n_trees=32, depth=4, dropout=0.0):
        super().__init__()
        self.n_trees = n_trees
        self.depth = depth
        n_leaves = 2 ** depth
        self.feat_weights = nn.Parameter(torch.randn(n_trees, depth, d_in) * 0.01)
        self.thresholds = nn.Parameter(torch.zeros(n_trees, depth))
        self.leaf_values = nn.Parameter(torch.randn(n_trees, n_leaves) * 0.01)
        self.drop = nn.Dropout(dropout)
        patterns = torch.zeros(n_leaves, depth)
        for i in range(n_leaves):
            for d in range(depth):
                patterns[i, d] = (i >> (depth - 1 - d)) & 1
        self.register_buffer("patterns", patterns)
    def forward(self, x):
        choices = torch.sigmoid(torch.einsum('bj,tdj->btd', x, self.feat_weights) - self.thresholds)
        c = choices.unsqueeze(2)
        p = self.patterns.unsqueeze(0).unsqueeze(0)
        leaf_probs = (c * p + (1 - c) * (1 - p)).prod(dim=-1)
        tree_out = (leaf_probs * self.leaf_values.unsqueeze(0)).sum(dim=-1)
        return self.drop(tree_out).mean(dim=-1, keepdim=True)


# ── 2.17  GRANDE ─────────────────────────────────────────────

class GRANDE(nn.Module):
    def __init__(self, d_in, n_trees=32, depth=4, dropout=0.0):
        super().__init__()
        self.n_trees = n_trees
        self.depth = depth
        n_leaves = 2 ** depth
        self.feat_logits = nn.Parameter(torch.randn(n_trees, depth, d_in) * 0.1)
        self.thresholds = nn.Parameter(torch.zeros(n_trees, depth))
        self.leaf_values = nn.Parameter(torch.randn(n_trees, n_leaves) * 0.01)
        self.tree_weights = nn.Parameter(torch.ones(n_trees) / n_trees)
        self.drop = nn.Dropout(dropout)
        patterns = torch.zeros(n_leaves, depth)
        for i in range(n_leaves):
            for d in range(depth):
                patterns[i, d] = (i >> (depth - 1 - d)) & 1
        self.register_buffer("patterns", patterns)
    def forward(self, x):
        feat_sel = F.softmax(self.feat_logits, dim=-1)
        proj = torch.einsum('bj,tdj->btd', x, feat_sel)
        choices = torch.sigmoid(10.0 * (proj - self.thresholds))
        c = choices.unsqueeze(2)
        p = self.patterns.unsqueeze(0).unsqueeze(0)
        leaf_probs = (c * p + (1 - c) * (1 - p)).prod(dim=-1)
        tree_out = (leaf_probs * self.leaf_values.unsqueeze(0)).sum(dim=-1)
        w = F.softmax(self.tree_weights, dim=0)
        return self.drop((tree_out * w).sum(dim=-1, keepdim=True))


# ── 2.18  Net-DNF ────────────────────────────────────────────

class NetDNF(nn.Module):
    def __init__(self, d_in, n_formulas=64, n_conjuncts=6, dropout=0.2):
        super().__init__()
        self.feat_sel = nn.Parameter(torch.randn(n_formulas, n_conjuncts, d_in) * 0.1)
        self.thresholds = nn.Parameter(torch.zeros(n_formulas, n_conjuncts))
        self.formula_w = nn.Linear(n_formulas, 1)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        sel = torch.sigmoid(self.feat_sel)
        proj = torch.einsum('bj,fcj->bfc', x, sel)
        conjuncts = torch.sigmoid(proj - self.thresholds)
        formulas = conjuncts.prod(dim=-1)
        return self.formula_w(self.drop(formulas))


# ── 2.19  TabNet (simplified) ───────────────────────────────

class TabNet(nn.Module):
    def __init__(self, d_in, n_steps=3, n_d=64, n_a=64, relax=1.5, dropout=0.1):
        super().__init__()
        self.n_steps = n_steps
        self.relax = relax
        self.bn_init = nn.BatchNorm1d(d_in)
        self.shared_fc = nn.Linear(d_in, n_d + n_a)
        self.step_fcs = nn.ModuleList([nn.Linear(d_in, n_d + n_a) for _ in range(n_steps)])
        self.attn_fcs = nn.ModuleList([nn.Sequential(nn.Linear(n_a, d_in), nn.BatchNorm1d(d_in)) for _ in range(n_steps)])
        self.head = nn.Linear(n_d, 1)
        self.n_d = n_d
        self.drop = nn.Dropout(dropout)
        self.aux_loss = torch.tensor(0.0)
    def forward(self, x):
        x = self.bn_init(x)
        prior_scales = torch.ones(x.shape[0], x.shape[1], device=x.device)
        agg = torch.zeros(x.shape[0], self.n_d, device=x.device)
        entropy_loss = 0.0
        for i in range(self.n_steps):
            h = self.shared_fc(x * prior_scales) + self.step_fcs[i](x * prior_scales)
            h_d, h_a = h[:, :self.n_d], h[:, self.n_d:]
            h_d = F.silu(h_d)
            agg = agg + self.drop(h_d)
            attn = self.attn_fcs[i](h_a)
            attn = F.softmax(attn * prior_scales, dim=-1)
            prior_scales = prior_scales * (self.relax - attn)
            entropy_loss += (-attn * torch.log(attn + 1e-15)).mean()
        if self.training:
            self.aux_loss = entropy_loss / self.n_steps
        return self.head(agg)


# ── 2.20  TabR ───────────────────────────────────────────────

class TabR(nn.Module):
    def __init__(self, d_in, hidden=256, n_layers=3, k=32, dropout=0.3):
        super().__init__()
        self.k = k
        layers = []
        prev = d_in
        for i in range(n_layers):
            h = hidden if i == 0 else hidden // 2
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.SiLU(), nn.Dropout(dropout)]
            prev = h
        self.encoder = nn.Sequential(*layers)
        self.latent_dim = prev
        self.retrieval_head = nn.Sequential(nn.Linear(prev * 2, 128), nn.SiLU(), nn.Dropout(dropout), nn.Linear(128, 1))
        self.direct_head = nn.Linear(prev, 1)
        self._store_z = None
        self._store_y = None
    def build_store(self, x_train, y_train):
        self.eval()
        with torch.no_grad():
            self._store_z = self.encoder(x_train)
            self._store_y = y_train
    def forward(self, x):
        z = self.encoder(x)
        if self._store_z is not None and not self.training:
            dists = torch.cdist(z, self._store_z)
            _, idx = dists.topk(self.k, largest=False)
            nbr_z = self._store_z[idx]
            sim = -dists.gather(1, idx)
            attn = torch.softmax(sim, dim=1).unsqueeze(-1)
            context = (attn * nbr_z).sum(dim=1)
            return self.retrieval_head(torch.cat([z, context], dim=1))
        return self.direct_head(z)


# ── 2.21  GrowNet stages ────────────────────────────────────

class GrowNetStage(nn.Module):
    def __init__(self, d_in, hidden=32, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in + 1, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, 1))
    def forward(self, x, prev_pred):
        return self.net(torch.cat([x, prev_pred], dim=1))


# ── 2.22  SwitchTab ──────────────────────────────────────────

class SwitchTab(nn.Module):
    def __init__(self, d_in, d_enc=128, d_mutual=64, d_salient=64, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(d_in, d_enc), nn.BatchNorm1d(d_enc), nn.SiLU(), nn.Dropout(dropout))
        self.mutual_proj = nn.Linear(d_enc, d_mutual)
        self.salient_proj = nn.Linear(d_enc, d_salient)
        self.decoder = nn.Sequential(nn.Linear(d_mutual + d_salient, d_in))
        self.head = nn.Sequential(nn.Linear(d_mutual + d_salient, 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
        self.aux_loss = torch.tensor(0.0)
    def forward(self, x):
        h = self.encoder(x)
        mutual = self.mutual_proj(h)
        salient = self.salient_proj(h)
        rep = torch.cat([mutual, salient], dim=1)
        if self.training:
            self.aux_loss = F.mse_loss(self.decoder(rep), x)
        return self.head(rep)


# ── 2.23  PTaRL ──────────────────────────────────────────────

class PTaRL(nn.Module):
    def __init__(self, d_in, n_prototypes=16, d_latent=128, hidden=256, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, d_latent), nn.BatchNorm1d(d_latent), nn.SiLU())
        self.prototypes = nn.Parameter(torch.randn(n_prototypes, d_latent) * 0.1)
        self.head = nn.Sequential(nn.Linear(d_latent * 2, 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        z = self.encoder(x)
        sim = F.cosine_similarity(z.unsqueeze(1), self.prototypes.unsqueeze(0), dim=-1)
        attn = F.softmax(sim * 10, dim=1)
        proto_rep = (attn.unsqueeze(-1) * self.prototypes.unsqueeze(0)).sum(dim=1)
        return self.head(torch.cat([z, proto_rep], dim=1))


# ── 2.24  DCN v2 ────────────────────────────────────────────

class CrossLayer(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.W = nn.Linear(d, d, bias=False)
        self.b = nn.Parameter(torch.zeros(d))
    def forward(self, x0, x):
        return x0 * self.W(x) + self.b + x

class DCNv2(nn.Module):
    def __init__(self, d_in, n_cross=3, hidden=256, dropout=0.3):
        super().__init__()
        self.cross_layers = nn.ModuleList([CrossLayer(d_in) for _ in range(n_cross)])
        self.deep = nn.Sequential(
            nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.BatchNorm1d(hidden // 2), nn.SiLU(), nn.Dropout(dropout))
        self.head = nn.Linear(d_in + hidden // 2, 1)
    def forward(self, x):
        x_cross = x
        for cl in self.cross_layers:
            x_cross = cl(x, x_cross)
        return self.head(torch.cat([x_cross, self.deep(x)], dim=1))


# ═══════════════════════════════════════════════════════════════
#  3. ФУНКЦИИ ОБУЧЕНИЯ
# ═══════════════════════════════════════════════════════════════

def train_model(model, epochs=300, lr=1e-3, wd=1e-4, bs=256,
                patience=30, aux_w=0.0, trial=None):
    """Обучение с early stopping. Возвращает (model, val_mae, epochs_used)."""
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", patience=10, factor=0.5, min_lr=1e-6)
    loss_fn = nn.L1Loss()
    ds = TensorDataset(X_tr_t, y_tr_t)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    X_v, y_v = X_va_t.to(device), y_va_t.to(device)

    best_val, best_state, wait = float("inf"), None, 0
    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            if aux_w > 0 and hasattr(model, "aux_loss"):
                loss = loss + aux_w * model.aux_loss
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            v_mae = loss_fn(model(X_v), y_v).item()
        sched.step(v_mae)
        if v_mae < best_val:
            best_val = v_mae
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
        if trial is not None:
            trial.report(v_mae, ep)
            if trial.should_prune():
                raise optuna.TrialPruned()
    model.load_state_dict(best_state)
    return model, best_val, ep


def train_grownet(n_stages=5, hidden=32, lr=0.01, wd=1e-4,
                  step_size=0.1, bs=256, patience=30, dropout=0.1, trial=None):
    """Gradient boosting с NN base learners. Возвращает (stages, val_mae, total_epochs)."""
    stages = []
    X_v, y_v = X_va_t.to(device), y_va_t.to(device)
    ds = TensorDataset(X_tr_t, y_tr_t)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    loss_fn = nn.L1Loss()

    # Предсказание ансамбля на train/val
    def ensemble_pred(x, stgs):
        p = torch.zeros(x.size(0), 1, device=x.device)
        for s in stgs:
            s.eval()
            p = p + step_size * s(x, p.detach())
        return p

    best_val, best_n = float("inf"), 0
    total_ep = 0
    for stage_idx in range(n_stages):
        model = GrowNetStage(INPUT_DIM, hidden, dropout).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", patience=10, factor=0.5, min_lr=1e-6)
        best_stage_val, best_stage_state, wait = float("inf"), None, 0
        for ep in range(1, 201):
            model.train()
            for xb, yb in dl:
                xb, yb = xb.to(device), yb.to(device)
                with torch.no_grad():
                    prev = ensemble_pred(xb, stages)
                correction = model(xb, prev)
                loss = loss_fn(prev + step_size * correction, yb)
                opt.zero_grad(); loss.backward(); opt.step()
            model.eval()
            with torch.no_grad():
                prev_v = ensemble_pred(X_v, stages)
                v_pred = prev_v + step_size * model(X_v, prev_v)
                v_mae = loss_fn(v_pred, y_v).item()
            sched.step(v_mae)
            if v_mae < best_stage_val:
                best_stage_val = v_mae
                best_stage_state = {k: v.clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1
                if wait >= patience:
                    break
            total_ep += 1
        model.load_state_dict(best_stage_state)
        stages.append(model)
        # Проверяем ансамбль
        with torch.no_grad():
            ens_val = loss_fn(ensemble_pred(X_v, stages), y_v).item()
        if ens_val < best_val:
            best_val = ens_val
            best_n = len(stages)
        if trial is not None:
            trial.report(ens_val, stage_idx)
            if trial.should_prune():
                raise optuna.TrialPruned()
    stages = stages[:best_n]
    return stages, best_val, total_ep


def eval_on_test(model, name=""):
    """Evaluate model on holdout test."""
    model.eval()
    with torch.no_grad():
        pred = model(X_te_t.to(device)).cpu().numpy().flatten()
    mae = mean_absolute_error(ytest, pred)
    rmse = np.sqrt(mean_squared_error(ytest, pred))
    mdae = median_absolute_error(ytest, pred)
    return mae, rmse, mdae


def eval_grownet_on_test(stages, step_size=0.1):
    """Evaluate GrowNet ensemble on holdout test."""
    X_t = X_te_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    mae = mean_absolute_error(ytest, pred)
    rmse = np.sqrt(mean_squared_error(ytest, pred))
    mdae = median_absolute_error(ytest, pred)
    return mae, rmse, mdae


print("24 архитектуры + train_model + train_grownet определены.")


In [ ]:
torch.manual_seed(seed)
np.random.seed(seed)

# ═══════════════════════════════════════════════════════════════
#  Baseline: все 24 архитектуры с дефолтными параметрами
# ═══════════════════════════════════════════════════════════════

DEFAULTS = {
    # MLP-семейство
    "MLP":            lambda: MLP(INPUT_DIM, [256, 128], dropout=0.3),
    "ResMLP":         lambda: ResMLP(INPUT_DIM, hidden=256, n_blocks=3, dropout=0.3),
    "SNN":            lambda: SNN(INPUT_DIM, hidden_dims=[256, 128], alpha_dropout=0.1),
    "GatedMLP":       lambda: GatedMLP(INPUT_DIM, hidden=256, n_blocks=3, dropout=0.3),
    "GANDALF":        lambda: GANDALF(INPUT_DIM, hidden=256, n_blocks=3, dropout=0.3),
    "DAE-MLP":        lambda: DAEMLP(INPUT_DIM, hidden=256, latent=64, noise=0.3, dropout=0.3),
    # CNN / RNN
    "CNN1D":          lambda: CNN1D(INPUT_DIM, channels=[64, 128, 64], ks=5, dropout=0.3),
    "BiGRU":          lambda: BiGRU(INPUT_DIM, hidden=64, n_layers=2, dropout=0.3),
    # Transformer / Attention
    "FT-Transformer": lambda: FTTransformer(INPUT_DIM, d_model=32, n_heads=4, n_layers=2, dropout=0.2),
    "TabTransformer": lambda: TabTransformer(INPUT_DIM, d_model=32, n_heads=4, n_layers=2, mlp_hidden=128, dropout=0.2),
    "SAINT":          lambda: SAINT(INPUT_DIM, d_model=32, n_heads=4, n_layers=2, dropout=0.2),
    "AutoInt":        lambda: AutoInt(INPUT_DIM, d_model=32, n_heads=4, n_layers=3, dropout=0.2),
    "Trompt":         lambda: Trompt(INPUT_DIM, d_model=32, n_heads=4, n_layers=3, dropout=0.2),
    "ExcelFormer":    lambda: ExcelFormer(INPUT_DIM, d_model=32, n_heads=4, n_layers=2, dropout=0.2),
    "ARM-Net":        lambda: ARMNet(INPUT_DIM, n_exp=64, hidden=128, order=2, dropout=0.2),
    # Tree-inspired
    "NODE":           lambda: NODE(INPUT_DIM, n_trees=32, depth=4, dropout=0.0),
    "GRANDE":         lambda: GRANDE(INPUT_DIM, n_trees=32, depth=4, dropout=0.0),
    "Net-DNF":        lambda: NetDNF(INPUT_DIM, n_formulas=64, n_conjuncts=6, dropout=0.2),
    "TabNet":         lambda: TabNet(INPUT_DIM, n_steps=3, n_d=64, n_a=64, relax=1.5, dropout=0.1),
    # Retrieval / Special
    "TabR":           lambda: TabR(INPUT_DIM, hidden=256, n_layers=3, k=32, dropout=0.3),
    "SwitchTab":      lambda: SwitchTab(INPUT_DIM, d_enc=128, d_mutual=64, d_salient=64, dropout=0.3),
    "PTaRL":          lambda: PTaRL(INPUT_DIM, n_prototypes=16, d_latent=128, hidden=256, dropout=0.3),
    "DCNv2":          lambda: DCNv2(INPUT_DIM, n_cross=3, hidden=256, dropout=0.3),
}

AUX_W = {"DAE-MLP": 0.1, "TabNet": 0.01, "SwitchTab": 0.1}
def build_arch_from_params(arch_name: str, bp: dict):
    if arch_name == "MLP":
        dims = [max(int(bp["first_hidden"]) // (2 ** i), 32) for i in range(int(bp["n_layers"]))]
        return MLP(INPUT_DIM, dims, bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "ResMLP":
        return ResMLP(INPUT_DIM, bp["hidden"], bp["n_blocks"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "SNN":
        dims = [max(int(bp["first_hidden"]) // (2 ** i), 32) for i in range(int(bp["n_layers"]))]
        return SNN(INPUT_DIM, dims, bp["alpha_dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "GatedMLP":
        return GatedMLP(INPUT_DIM, bp["hidden"], bp["n_blocks"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "GANDALF":
        return GANDALF(INPUT_DIM, bp["hidden"], bp["n_blocks"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "DAE-MLP":
        return DAEMLP(INPUT_DIM, bp["hidden"], bp["latent"], bp["noise"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "CNN1D":
        nc = int(bp["n_conv"]); bc = int(bp["base_ch"])
        chs = [bc * (2 ** i) for i in range(nc // 2 + 1)]
        chs += [bc * (2 ** i) for i in range(nc // 2 - 1, -1, -1)]
        chs = chs[:nc]
        return CNN1D(INPUT_DIM, chs, bp["ks"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "BiGRU":
        return BiGRU(INPUT_DIM, bp["hidden"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "FT-Transformer":
        return FTTransformer(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "TabTransformer":
        return TabTransformer(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["mlp_hidden"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "SAINT":
        return SAINT(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "AutoInt":
        return AutoInt(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "Trompt":
        return Trompt(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "ExcelFormer":
        return ExcelFormer(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "ARM-Net":
        return ARMNet(INPUT_DIM, bp["n_exp"], bp["hidden"], bp["order"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "NODE":
        return NODE(INPUT_DIM, bp["n_trees"], bp["depth"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "GRANDE":
        return GRANDE(INPUT_DIM, bp["n_trees"], bp["depth"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "Net-DNF":
        return NetDNF(INPUT_DIM, bp["n_formulas"], bp["n_conjuncts"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "TabNet":
        return TabNet(INPUT_DIM, bp["n_steps"], bp["n_d"], bp["n_a"], bp["relax"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "TabR":
        return TabR(INPUT_DIM, bp["hidden"], bp["n_layers"], bp["k"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "SwitchTab":
        return SwitchTab(INPUT_DIM, bp["d_enc"], bp["d_mutual"], bp["d_salient"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "PTaRL":
        return PTaRL(INPUT_DIM, bp["n_prototypes"], bp["d_latent"], bp["hidden"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "DCNv2":
        return DCNv2(INPUT_DIM, bp["n_cross"], bp["hidden"], bp["dropout"]), bp.get("aux_w", 0.0)
    else:
        raise KeyError(f"Unknown architecture: {arch_name}")

def serialize_params(params: dict):
    clean = {}
    for k, v in params.items():
        if isinstance(v, (np.integer,)):
            clean[k] = int(v)
        elif isinstance(v, (np.floating,)):
            clean[k] = float(v)
        else:
            clean[k] = v
    return clean

In [ ]:
torch.manual_seed(seed)
np.random.seed(seed)

# ═══════════════════════════════════════════════════════════════
#  Optuna objectives + baseline-aware honest tuning
# ═══════════════════════════════════════════════════════════════

BASELINE_TRIAL_PARAMS = {
    "MLP":            {"n_layers": 2, "first_hidden": 256, "dropout": 0.3, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "ResMLP":         {"hidden": 256, "n_blocks": 3, "dropout": 0.3, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "SNN":            {"n_layers": 2, "first_hidden": 256, "alpha_dropout": 0.1, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "GatedMLP":       {"hidden": 256, "n_blocks": 3, "dropout": 0.3, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "GANDALF":        {"hidden": 256, "n_blocks": 3, "dropout": 0.3, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "DAE-MLP":        {"hidden": 256, "latent": 64, "noise": 0.3, "dropout": 0.3, "aux_w": 0.1, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "CNN1D":          {"n_conv": 3, "base_ch": 64, "ks": 5, "dropout": 0.3, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "BiGRU":          {"hidden": 64, "n_layers": 2, "dropout": 0.3, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "FT-Transformer": {"d_model": 32, "n_heads": 4, "n_layers": 2, "dropout": 0.2, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "TabTransformer": {"d_model": 32, "n_heads": 4, "n_layers": 2, "mlp_hidden": 128, "dropout": 0.2, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "SAINT":          {"d_model": 32, "n_heads": 4, "n_layers": 2, "dropout": 0.2, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "AutoInt":        {"d_model": 32, "n_heads": 4, "n_layers": 3, "dropout": 0.2, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "Trompt":         {"d_model": 32, "n_heads": 4, "n_layers": 3, "dropout": 0.2, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "ExcelFormer":    {"d_model": 32, "n_heads": 4, "n_layers": 2, "dropout": 0.2, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "ARM-Net":        {"n_exp": 64, "hidden": 128, "order": 2, "dropout": 0.2, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "NODE":           {"n_trees": 32, "depth": 4, "dropout": 0.0, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "GRANDE":         {"n_trees": 32, "depth": 4, "dropout": 0.0, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "Net-DNF":        {"n_formulas": 64, "n_conjuncts": 6, "dropout": 0.2, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "TabNet":         {"n_steps": 3, "n_d": 64, "n_a": 64, "relax": 1.5, "dropout": 0.1, "aux_w": 0.01, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "TabR":           {"hidden": 256, "n_layers": 3, "k": 32, "dropout": 0.3, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "SwitchTab":      {"d_enc": 128, "d_mutual": 64, "d_salient": 64, "dropout": 0.3, "aux_w": 0.1, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "PTaRL":          {"n_prototypes": 16, "d_latent": 128, "hidden": 256, "dropout": 0.3, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "DCNv2":          {"n_cross": 3, "hidden": 256, "dropout": 0.3, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "GrowNet":        {"n_stages": 5, "hidden": 32, "step_size": 0.1, "dropout": 0.1, "lr": 0.01, "wd": 1e-4, "bs": 256},
    "TabPFN":         {"n_estimators": 8, "max_rows": 3000},
}

def _seed_trial():
    # Один и тот же seed для каждого trial -> сравнение гиперпараметров честнее и ближе к baseline.
    seed_everything(SEED)

def _suggest_opt(trial, *, bs_choices, lr_low=1e-4, lr_high=3e-3, wd_low=1e-5, wd_high=1e-3):
    lr = trial.suggest_float("lr", lr_low, lr_high, log=True)
    wd = trial.suggest_float("wd", wd_low, wd_high, log=True)
    bs = trial.suggest_categorical("bs", bs_choices)
    return lr, wd, bs

def obj_mlp(trial):
    _seed_trial()
    n_layers = trial.suggest_int("n_layers", 2, 4)
    first = trial.suggest_categorical("first_hidden", [128, 256, 512])
    dims = [max(first // (2**i), 32) for i in range(n_layers)]
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(MLP(INPUT_DIM, dims, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_resmlp(trial):
    _seed_trial()
    hidden = trial.suggest_categorical("hidden", [128, 256, 384])
    n_blocks = trial.suggest_int("n_blocks", 2, 5)
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(ResMLP(INPUT_DIM, hidden, n_blocks, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_snn(trial):
    _seed_trial()
    n_layers = trial.suggest_int("n_layers", 2, 4)
    first = trial.suggest_categorical("first_hidden", [128, 256, 512])
    dims = [max(first // (2**i), 32) for i in range(n_layers)]
    alpha_do = trial.suggest_float("alpha_dropout", 0.01, 0.2)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512], wd_low=1e-6)
    return train_model(SNN(INPUT_DIM, dims, alpha_do), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_gatedmlp(trial):
    _seed_trial()
    hidden = trial.suggest_categorical("hidden", [128, 256, 384])
    n_blocks = trial.suggest_int("n_blocks", 2, 5)
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(GatedMLP(INPUT_DIM, hidden, n_blocks, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_gandalf(trial):
    _seed_trial()
    hidden = trial.suggest_categorical("hidden", [128, 256, 384])
    n_blocks = trial.suggest_int("n_blocks", 2, 5)
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(GANDALF(INPUT_DIM, hidden, n_blocks, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_daemlp(trial):
    _seed_trial()
    hidden = trial.suggest_categorical("hidden", [128, 256, 512])
    latent = trial.suggest_categorical("latent", [32, 64, 128])
    noise = trial.suggest_float("noise", 0.1, 0.5)
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    aux_w = trial.suggest_float("aux_w", 0.01, 0.5, log=True)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(DAEMLP(INPUT_DIM, hidden, latent, noise, dropout), lr=lr, wd=wd, bs=bs, aux_w=aux_w, trial=trial)[1]

def obj_cnn1d(trial):
    _seed_trial()
    n_conv = trial.suggest_int("n_conv", 2, 4)
    base_ch = trial.suggest_categorical("base_ch", [32, 64, 128])
    chs = [base_ch * (2**i) for i in range(n_conv // 2 + 1)]
    chs += [base_ch * (2**i) for i in range(n_conv // 2 - 1, -1, -1)]
    chs = chs[:n_conv]
    ks = trial.suggest_categorical("ks", [3, 5, 7, 9])
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(CNN1D(INPUT_DIM, chs, ks, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_bigru(trial):
    _seed_trial()
    hidden = trial.suggest_categorical("hidden", [32, 64, 128])
    n_layers = trial.suggest_int("n_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256], wd_low=1e-5)
    return train_model(BiGRU(INPUT_DIM, hidden, n_layers, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_ft_transformer(trial):
    _seed_trial()
    d_model = trial.suggest_categorical("d_model", [16, 32, 48])
    n_heads = trial.suggest_categorical("n_heads", [2, 4])
    n_layers = trial.suggest_int("n_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.05, 0.3)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[64, 128, 256], lr_low=5e-5, lr_high=1e-3, wd_low=1e-6)
    return train_model(FTTransformer(INPUT_DIM, d_model, n_heads, n_layers, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_tabtransformer(trial):
    _seed_trial()
    d_model = trial.suggest_categorical("d_model", [16, 32])
    n_heads = trial.suggest_categorical("n_heads", [2, 4])
    n_layers = trial.suggest_int("n_layers", 1, 3)
    mlp_hidden = trial.suggest_categorical("mlp_hidden", [64, 128, 256])
    dropout = trial.suggest_float("dropout", 0.05, 0.3)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[64, 128, 256], lr_low=5e-5, lr_high=1e-3, wd_low=1e-6)
    return train_model(TabTransformer(INPUT_DIM, d_model, n_heads, n_layers, mlp_hidden, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_saint(trial):
    _seed_trial()
    d_model = trial.suggest_categorical("d_model", [16, 32])
    n_heads = trial.suggest_categorical("n_heads", [2, 4])
    n_layers = trial.suggest_int("n_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.05, 0.3)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[64, 128, 256], lr_low=5e-5, lr_high=1e-3, wd_low=1e-6)
    return train_model(SAINT(INPUT_DIM, d_model, n_heads, n_layers, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_autoint(trial):
    _seed_trial()
    d_model = trial.suggest_categorical("d_model", [8, 16, 32])
    n_heads = trial.suggest_categorical("n_heads", [2, 4])
    n_layers = trial.suggest_int("n_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.05, 0.3)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[64, 128, 256], lr_low=5e-5, lr_high=1e-3, wd_low=1e-6)
    return train_model(AutoInt(INPUT_DIM, d_model, n_heads, n_layers, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_trompt(trial):
    _seed_trial()
    d_model = trial.suggest_categorical("d_model", [16, 32])
    n_heads = trial.suggest_categorical("n_heads", [2, 4])
    n_layers = trial.suggest_int("n_layers", 2, 4)
    dropout = trial.suggest_float("dropout", 0.05, 0.3)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[64, 128, 256], lr_low=5e-5, lr_high=1e-3, wd_low=1e-6)
    return train_model(Trompt(INPUT_DIM, d_model, n_heads, n_layers, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_excelformer(trial):
    _seed_trial()
    d_model = trial.suggest_categorical("d_model", [16, 32])
    n_heads = trial.suggest_categorical("n_heads", [2, 4])
    n_layers = trial.suggest_int("n_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.05, 0.3)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[64, 128, 256], lr_low=5e-5, lr_high=1e-3, wd_low=1e-6)
    return train_model(ExcelFormer(INPUT_DIM, d_model, n_heads, n_layers, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_armnet(trial):
    _seed_trial()
    n_exp = trial.suggest_categorical("n_exp", [32, 64, 128])
    hidden = trial.suggest_categorical("hidden", [64, 128, 256])
    order = trial.suggest_int("order", 2, 3)
    dropout = trial.suggest_float("dropout", 0.1, 0.3)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256])
    return train_model(ARMNet(INPUT_DIM, n_exp, hidden, order, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_node(trial):
    _seed_trial()
    n_trees = trial.suggest_categorical("n_trees", [16, 32, 64, 128])
    depth = trial.suggest_int("depth", 3, 6)
    dropout = trial.suggest_float("dropout", 0.0, 0.3)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512], wd_low=1e-6)
    return train_model(NODE(INPUT_DIM, n_trees, depth, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_grande(trial):
    _seed_trial()
    n_trees = trial.suggest_categorical("n_trees", [16, 32, 64])
    depth = trial.suggest_int("depth", 3, 5)
    dropout = trial.suggest_float("dropout", 0.0, 0.2)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512], wd_low=1e-6)
    return train_model(GRANDE(INPUT_DIM, n_trees, depth, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_netdnf(trial):
    _seed_trial()
    n_formulas = trial.suggest_categorical("n_formulas", [32, 64, 128])
    n_conjuncts = trial.suggest_int("n_conjuncts", 4, 8)
    dropout = trial.suggest_float("dropout", 0.1, 0.3)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(NetDNF(INPUT_DIM, n_formulas, n_conjuncts, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_tabnet(trial):
    _seed_trial()
    n_steps = trial.suggest_int("n_steps", 2, 5)
    n_d = trial.suggest_categorical("n_d", [32, 64, 128])
    n_a = trial.suggest_categorical("n_a", [32, 64, 128])
    relax = trial.suggest_float("relax", 1.0, 2.0)
    dropout = trial.suggest_float("dropout", 0.01, 0.2)
    aux_w = trial.suggest_float("aux_w", 0.001, 0.1, log=True)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256])
    return train_model(TabNet(INPUT_DIM, n_steps, n_d, n_a, relax, dropout), lr=lr, wd=wd, bs=bs, aux_w=aux_w, trial=trial)[1]

def obj_tabr(trial):
    _seed_trial()
    hidden = trial.suggest_categorical("hidden", [128, 256, 384])
    n_layers = trial.suggest_int("n_layers", 2, 4)
    k = trial.suggest_categorical("k", [16, 32, 64])
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256])
    return train_model(TabR(INPUT_DIM, hidden, n_layers, k, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_switchtab(trial):
    _seed_trial()
    d_enc = trial.suggest_categorical("d_enc", [64, 128, 256])
    d_mutual = trial.suggest_categorical("d_mutual", [32, 64, 128])
    d_salient = trial.suggest_categorical("d_salient", [32, 64, 128])
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    aux_w = trial.suggest_float("aux_w", 0.01, 0.5, log=True)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(SwitchTab(INPUT_DIM, d_enc, d_mutual, d_salient, dropout), lr=lr, wd=wd, bs=bs, aux_w=aux_w, trial=trial)[1]

def obj_ptarl(trial):
    _seed_trial()
    n_proto = trial.suggest_categorical("n_prototypes", [8, 16, 32])
    d_latent = trial.suggest_categorical("d_latent", [64, 128, 256])
    hidden = trial.suggest_categorical("hidden", [128, 256, 384])
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(PTaRL(INPUT_DIM, n_proto, d_latent, hidden, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_dcnv2(trial):
    _seed_trial()
    n_cross = trial.suggest_int("n_cross", 2, 5)
    hidden = trial.suggest_categorical("hidden", [128, 256, 384])
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(DCNv2(INPUT_DIM, n_cross, hidden, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_grownet(trial):
    _seed_trial()
    n_stages = trial.suggest_int("n_stages", 3, 8)
    hidden = trial.suggest_categorical("hidden", [16, 32, 64])
    step_size = trial.suggest_float("step_size", 0.05, 0.3)
    dropout = trial.suggest_float("dropout", 0.0, 0.2)
    lr = trial.suggest_float("lr", 1e-3, 0.05, log=True)
    wd = trial.suggest_float("wd", 1e-5, 1e-3, log=True)
    bs = trial.suggest_categorical("bs", [128, 256, 512])
    _, val_mae, _ = train_grownet(n_stages, hidden, lr, wd, step_size, bs, patience=30, dropout=dropout, trial=trial)
    return val_mae

def obj_tabpfn(trial):
    """TabPFN: baseline-aware tuning on the same inner train/val split as baseline."""
    _seed_trial()
    n_est = trial.suggest_int("n_estimators", 2, 32)
    max_rows = trial.suggest_categorical("max_rows", [1000, 2000, 3000])
    pfn_device = "cuda" if torch.cuda.is_available() else "cpu"

    Xp = X_dl_tr.copy()
    yp = y_dl_tr.copy()
    if len(Xp) > max_rows:
        idx = np.random.RandomState(SEED).choice(len(Xp), max_rows, replace=False)
        Xp = Xp[idx]
        yp = yp[idx]
    np.nan_to_num(Xp, copy=False)

    Xv = X_dl_va.copy()
    yv = y_dl_va.copy()
    np.nan_to_num(Xv, copy=False)

    model = TabPFNRegressor(device=pfn_device, n_estimators=n_est)
    model.fit(Xp, yp)
    pred = model.predict(Xv)
    return mean_absolute_error(yv, pred)

In [ ]:
def clear_device_cache():
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps":
        torch.mps.empty_cache()

def calc_reg_metrics(y_true, pred):
    return {
        "MAE": float(mean_absolute_error(y_true, pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, pred))),
        "MdAE": float(median_absolute_error(y_true, pred)),
    }

def eval_on_typical_holdout(model):
    model.eval()
    with torch.no_grad():
        pred = model(X_te_t.to(device)).cpu().numpy().flatten()
    return calc_reg_metrics(ytest_typical, pred)

def eval_on_full_holdout(model):
    model.eval()
    with torch.no_grad():
        pred = model(X_te_stress_t.to(device)).cpu().numpy().flatten()
    return calc_reg_metrics(ytest_full, pred)

def eval_grownet_on_typical_holdout(stages, step_size=0.1):
    X_t = X_te_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    return calc_reg_metrics(ytest_typical, pred)

def eval_grownet_on_full_holdout(stages, step_size=0.1):
    X_t = X_te_stress_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    return calc_reg_metrics(ytest_full, pred)

def eval_tabpfn_holdouts(tabpfn_model, use_full_train_scaler=False):
    if use_full_train_scaler:
        X_typ = X_te_full_t.numpy()
        X_full = X_te_fullscale_stress_t.numpy()
    else:
        X_typ = X_te_t.numpy()
        X_full = X_te_stress_t.numpy()
    pred_typ = tabpfn_model.predict(X_typ)
    pred_full = tabpfn_model.predict(X_full)
    return calc_reg_metrics(ytest_typical, pred_typ), calc_reg_metrics(ytest_full, pred_full)

def train_model_fulltrain(model, epochs=100, lr=1e-3, wd=1e-4, bs=256, aux_w=0.0):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1), eta_min=1e-6)
    loss_fn = nn.L1Loss()
    ds = TensorDataset(X_full_t, y_full_t)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    for ep in range(epochs):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            if aux_w > 0 and hasattr(model, "aux_loss"):
                loss = loss + aux_w * model.aux_loss
            opt.zero_grad()
            loss.backward()
            opt.step()
        sched.step()
    return model

def eval_fulltrain_model_typical(model):
    model.eval()
    with torch.no_grad():
        pred = model(X_te_full_t.to(device)).cpu().numpy().flatten()
    return calc_reg_metrics(ytest_typical, pred)

def eval_fulltrain_model_full(model):
    model.eval()
    with torch.no_grad():
        pred = model(X_te_fullscale_stress_t.to(device)).cpu().numpy().flatten()
    return calc_reg_metrics(ytest_full, pred)

def train_grownet_fulltrain(n_stages=5, hidden=32, lr=0.01, wd=1e-4,
                            step_size=0.1, bs=256, epochs_per_stage=50, dropout=0.1):
    stages = []
    ds = TensorDataset(X_full_t, y_full_t)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    loss_fn = nn.L1Loss()

    def ensemble_pred(x, stgs):
        p = torch.zeros(x.size(0), 1, device=x.device)
        for s in stgs:
            s.eval()
            p = p + step_size * s(x, p.detach())
        return p

    for stage_idx in range(n_stages):
        model = GrowNetStage(INPUT_DIM, hidden, dropout).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs_per_stage, 1), eta_min=1e-6)

        for ep in range(epochs_per_stage):
            model.train()
            for xb, yb in dl:
                xb, yb = xb.to(device), yb.to(device)
                with torch.no_grad():
                    prev = ensemble_pred(xb, stages)
                correction = model(xb, prev)
                loss = loss_fn(prev + step_size * correction, yb)
                opt.zero_grad()
                loss.backward()
                opt.step()
            sched.step()

        stages.append(model)
    return stages

def eval_fulltrain_grownet_typical(stages, step_size=0.1):
    X_t = X_te_full_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    return calc_reg_metrics(ytest_typical, pred)

def eval_fulltrain_grownet_full(stages, step_size=0.1):
    X_t = X_te_fullscale_stress_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    return calc_reg_metrics(ytest_full, pred)

print("Full-holdout helpers defined.")

In [ ]:
RUN_TABPFN = RUN_TABPFN_DEFAULT and HAS_TABPFN

optuna.logging.set_verbosity(optuna.logging.WARNING)

OBJECTIVE_MAP = {
    "MLP": obj_mlp,
    "ResMLP": obj_resmlp,
    "SNN": obj_snn,
    "GatedMLP": obj_gatedmlp,
    "GANDALF": obj_gandalf,
    "DAE-MLP": obj_daemlp,
    "CNN1D": obj_cnn1d,
    "BiGRU": obj_bigru,
    "FT-Transformer": obj_ft_transformer,
    "TabTransformer": obj_tabtransformer,
    "SAINT": obj_saint,
    "AutoInt": obj_autoint,
    "Trompt": obj_trompt,
    "ExcelFormer": obj_excelformer,
    "ARM-Net": obj_armnet,
    "NODE": obj_node,
    "GRANDE": obj_grande,
    "Net-DNF": obj_netdnf,
    "TabNet": obj_tabnet,
    "TabR": obj_tabr,
    "SwitchTab": obj_switchtab,
    "PTaRL": obj_ptarl,
    "DCNv2": obj_dcnv2,
    "GrowNet": obj_grownet,
}
if RUN_TABPFN and HAS_TABPFN:
    OBJECTIVE_MAP["TabPFN"] = obj_tabpfn

def fit_best_inner_model(arch_name: str, bp: dict):
    seed_everything(SEED)
    if arch_name == "GrowNet":
        stages, val_mae, epochs_used = train_grownet(
            n_stages=bp["n_stages"],
            hidden=bp["hidden"],
            lr=bp["lr"],
            wd=bp["wd"],
            step_size=bp["step_size"],
            bs=bp["bs"],
            patience=30,
            dropout=bp["dropout"],
        )
        metrics_typ = eval_grownet_on_typical_holdout(stages, bp["step_size"])
        metrics_full = eval_grownet_on_full_holdout(stages, bp["step_size"])
        n_params = int(sum(sum(p.numel() for p in s.parameters()) for s in stages))
        return {
            "kind": "grownet",
            "fitted": stages,
            "val_mae": float(val_mae),
            "epochs_used": int(epochs_used),
            "n_params": n_params,
            "metrics_typical": metrics_typ,
            "metrics_full": metrics_full,
        }

    if arch_name == "TabPFN":
        pfn_device = "cuda" if torch.cuda.is_available() else "cpu"
        max_rows = bp.get("max_rows", 3000)
        n_est = bp.get("n_estimators", 8)
        X_pfn_tr = X_dl_tr.copy()
        y_pfn_tr = y_dl_tr.copy()
        if len(X_pfn_tr) > max_rows:
            idx = np.random.RandomState(SEED).choice(len(X_pfn_tr), max_rows, replace=False)
            X_pfn_tr = X_pfn_tr[idx]
            y_pfn_tr = y_pfn_tr[idx]
        tabpfn = TabPFNRegressor(device=pfn_device, n_estimators=n_est)
        tabpfn.fit(X_pfn_tr, y_pfn_tr)
        val_pred = tabpfn.predict(X_dl_va)
        val_mae = mean_absolute_error(y_dl_va, val_pred)
        metrics_typ, metrics_full = eval_tabpfn_holdouts(tabpfn_model=tabpfn, use_full_train_scaler=False)
        return {
            "kind": "tabpfn",
            "fitted": tabpfn,
            "val_mae": float(val_mae),
            "epochs_used": 0,
            "n_params": 0,
            "metrics_typical": metrics_typ,
            "metrics_full": metrics_full,
        }

    model, aux_w = build_arch_from_params(arch_name, bp)
    model, val_mae, epochs_used = train_model(
        model,
        epochs=300,
        lr=bp["lr"],
        wd=bp["wd"],
        bs=bp["bs"],
        patience=30,
        aux_w=aux_w,
    )
    if arch_name == "TabR":
        model.build_store(X_tr_t.to(device), y_tr_t.to(device))
    metrics_typ = eval_on_typical_holdout(model)
    metrics_full = eval_on_full_holdout(model)
    n_params = int(sum(p.numel() for p in model.parameters()))
    return {
        "kind": "torch",
        "fitted": model,
        "val_mae": float(val_mae),
        "epochs_used": int(epochs_used),
        "n_params": n_params,
        "metrics_typical": metrics_typ,
        "metrics_full": metrics_full,
    }



In [ ]:

# ═══════════════════════════════════════════════════════════════
#  Per-variant DL setup + tuning runner
# ═══════════════════════════════════════════════════════════════
import json as _json

DL_RUN_ARCHS = list(OBJECTIVE_MAP.keys())
# Как в DL_tuning.ipynb (ARCH_TRIALS = {k:50 for k in ARCH_TRIALS_DEFAULT})
ARCH_TRIALS_DEFAULT = {
    "MLP": 40, "SNN": 40, "GatedMLP": 40, "GANDALF": 40, "DCNv2": 40,
    "ResMLP": 30, "CNN1D": 30, "BiGRU": 30, "DAE-MLP": 30, "TabNet": 30,
    "Net-DNF": 30, "ARM-Net": 30, "PTaRL": 30, "SwitchTab": 30, "TabR": 30,
    "FT-Transformer": 25, "TabTransformer": 25, "SAINT": 25, "AutoInt": 25,
    "Trompt": 25, "ExcelFormer": 25, "NODE": 25, "GRANDE": 25,
    "GrowNet": 20, "TabPFN": 20,
}
DL_TRIALS_PER_ARCH = int(os.getenv("DL_TRIALS_PER_ARCH", "50"))
ARCH_TRIALS = {k: DL_TRIALS_PER_ARCH for k in ARCH_TRIALS_DEFAULT}
DL_SMOKE_TEST = os.getenv("DL_SMOKE_TEST", "0") == "1"
if DL_SMOKE_TEST:
    DL_RUN_ARCHS = ["MLP", "FT-Transformer"]
    ARCH_TRIALS = {k: 2 for k in ARCH_TRIALS_DEFAULT}
RUN_EXISTING_RESULTS_ONLY = os.getenv("RUN_EXISTING_RESULTS_ONLY", "0") == "1"
DL_INNER_VAL_FRAC = 0.2

def setup_dl_for_variant(Xtr_aug, ytr, Xte_typ_aug, yte_typ, Xte_full_aug, yte_full):
    global X_dl_tr, y_dl_tr, X_dl_va, y_dl_va, X_dl_te, X_full_s, X_te_full_s
    global X_tr_t, y_tr_t, X_va_t, y_va_t, X_te_t, X_te_stress_t
    global X_full_t, y_full_t, X_te_full_t, X_te_fullscale_stress_t
    global INPUT_DIM, sc, sc_full
    global Xtrain, ytrain, Xtest, ytest, ytest_typical, ytest_full

    Xtrain = Xtr_aug.copy()
    ytrain = pd.Series(ytr).reset_index(drop=True)
    Xtest = Xte_typ_aug.copy()
    ytest = pd.Series(yte_typ).reset_index(drop=True)
    ytest_typical = ytest.copy()
    ytest_full = pd.Series(yte_full).reset_index(drop=True)

    val_cut = int(len(Xtrain) * (1 - DL_INNER_VAL_FRAC))
    Xtr_arr = Xtrain.values.astype(np.float32) if hasattr(Xtrain, "values") else np.asarray(Xtrain, dtype=np.float32)
    ytr_arr = ytrain.values.astype(np.float32) if hasattr(ytrain, "values") else np.asarray(ytrain, dtype=np.float32)
    Xte_typ_arr = Xtest.values.astype(np.float32) if hasattr(Xtest, "values") else np.asarray(Xtest, dtype=np.float32)
    Xte_full_arr = Xte_full_aug.values.astype(np.float32) if hasattr(Xte_full_aug, "values") else np.asarray(Xte_full_aug, dtype=np.float32)

    X_dl_tr = Xtr_arr[:val_cut]
    y_dl_tr = ytr_arr[:val_cut]
    X_dl_va = Xtr_arr[val_cut:]
    y_dl_va = ytr_arr[val_cut:]

    sc = StandardScaler()
    X_dl_tr = sc.fit_transform(X_dl_tr)
    X_dl_va = sc.transform(X_dl_va)
    X_dl_te = sc.transform(Xte_typ_arr)
    X_dl_te_full = sc.transform(Xte_full_arr)

    sc_full = StandardScaler()
    X_full_s = sc_full.fit_transform(Xtr_arr)
    X_te_full_s = sc_full.transform(Xte_typ_arr)
    X_te_fullscale_stress = sc_full.transform(Xte_full_arr)

    for arr in [X_dl_tr, X_dl_va, X_dl_te, X_dl_te_full, X_full_s, X_te_full_s, X_te_fullscale_stress]:
        np.nan_to_num(arr, copy=False)

    X_tr_t = torch.from_numpy(X_dl_tr)
    y_tr_t = torch.from_numpy(y_dl_tr).unsqueeze(1)
    X_va_t = torch.from_numpy(X_dl_va)
    y_va_t = torch.from_numpy(y_dl_va).unsqueeze(1)
    X_te_t = torch.from_numpy(X_dl_te)
    X_te_stress_t = torch.from_numpy(X_dl_te_full)
    X_full_t = torch.from_numpy(X_full_s)
    y_full_t = torch.from_numpy(ytr_arr).unsqueeze(1)
    X_te_full_t = torch.from_numpy(X_te_full_s)
    X_te_fullscale_stress_t = torch.from_numpy(X_te_fullscale_stress)

    INPUT_DIM = X_dl_tr.shape[1]
    print(f"  setup_dl_for_variant: INPUT_DIM={INPUT_DIM}, train={X_dl_tr.shape}, val={X_dl_va.shape}, test_typ={X_dl_te.shape}, test_full={X_dl_te_full.shape}")

def _load_cached_rows(cache_path):
    cache_path = Path(cache_path) if cache_path is not None else None
    if cache_path is not None and cache_path.exists():
        cached = pd.read_csv(cache_path)
        print(f"  [RESUME] loaded {len(cached)} cached rows from {cache_path}")
        return cached.to_dict("records")
    return []


def _save_cached_rows(rows, cache_path):
    cache_path = Path(cache_path) if cache_path is not None else None
    if cache_path is None:
        return
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = cache_path.with_suffix(cache_path.suffix + ".tmp")
    out = pd.DataFrame(rows).sort_values(
        ["cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE"]
    ).reset_index(drop=True)
    out.to_csv(tmp_path, index=False)
    tmp_path.replace(cache_path)
    print(f"  [CACHE] saved {len(out)} rows -> {cache_path}")


def _safe_cache_name(text):
    import re as _re
    return _re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text)).strip("_") or "experiment"


def run_variant_dl_tuning(Xtr_aug, Xte_typ_aug, Xte_full_aug, title="experiment",
                          archs=None, n_trials_per_arch=None,
                          cache_path=None, study_dir=None):
    if archs is None:
        archs = DL_RUN_ARCHS
    if n_trials_per_arch is None:
        n_trials_per_arch = DL_TRIALS_PER_ARCH

    print("=" * 95)
    print(title)
    print("=" * 95)

    setup_dl_for_variant(
        Xtr_aug, meta_ytr0,
        Xte_typ_aug, meta_yte_typ0,
        Xte_full_aug, meta_yte0,
    )

    rows = _load_cached_rows(cache_path)
    done_models = {str(r.get("model")) for r in rows}
    if study_dir is not None:
        study_dir = Path(study_dir)
        study_dir.mkdir(parents=True, exist_ok=True)
    title_key = _safe_cache_name(title)

    for arch_name in archs:
        if arch_name in done_models:
            print(f"  [SKIP] {arch_name}: already cached in {cache_path}")
            continue
        if arch_name not in OBJECTIVE_MAP:
            print(f"  skip {arch_name}: no objective")
            continue
        arch_n_trials = ARCH_TRIALS.get(arch_name, n_trials_per_arch)
        t0 = time.time()
        seed_everything(SEED)
        sampler = optuna.samplers.TPESampler(seed=SEED, multivariate=True, warn_independent_sampling=False)
        pruner = optuna.pruners.MedianPruner(n_startup_trials=min(8, max(3, arch_n_trials // 3)), n_warmup_steps=20)

        storage = None
        study_name = None
        if study_dir is not None:
            study_name = f"{title_key}__{_safe_cache_name(arch_name)}"
            storage_path = study_dir / f"{study_name}.db"
            storage = f"sqlite:///{storage_path}"
        study = optuna.create_study(
            direction="minimize",
            sampler=sampler,
            pruner=pruner,
            storage=storage,
            study_name=study_name,
            load_if_exists=bool(storage),
        )
        finished_states = {optuna.trial.TrialState.COMPLETE, optuna.trial.TrialState.PRUNED, optuna.trial.TrialState.FAIL}
        finished_trials = [t for t in study.trials if t.state in finished_states]
        if len(study.trials) == 0 and arch_name in BASELINE_TRIAL_PARAMS:
            try:
                study.enqueue_trial(serialize_params(BASELINE_TRIAL_PARAMS[arch_name]))
            except Exception:
                pass
        remaining_trials = max(0, int(arch_n_trials) - len(finished_trials))
        print(f"  {arch_name}: cached_trials={len(finished_trials)}/{arch_n_trials}, remaining={remaining_trials}")
        try:
            if remaining_trials > 0:
                study.optimize(OBJECTIVE_MAP[arch_name], n_trials=remaining_trials, show_progress_bar=False)
        except Exception as e:
            print(f"  {arch_name}: optuna failed: {type(e).__name__}: {e}")
            _save_cached_rows(rows, cache_path)
            clear_device_cache()
            continue
        try:
            bp = serialize_params(study.best_params)
        except Exception as e:
            print(f"  {arch_name}: no completed Optuna trial: {type(e).__name__}: {e}")
            _save_cached_rows(rows, cache_path)
            clear_device_cache()
            continue
        try:
            fit = fit_best_inner_model(arch_name, bp)
        except Exception as e:
            print(f"  {arch_name}: refit failed: {type(e).__name__}: {e}")
            _save_cached_rows(rows, cache_path)
            clear_device_cache()
            continue
        rows.append({
            "model": arch_name,
            "cv_MAE_internal": round(float(study.best_value), 4),
            "refit_val_MAE": round(float(fit["val_mae"]), 4),
            "holdout_typical_MAE": round(float(fit["metrics_typical"]["MAE"]), 4),
            "holdout_full_MAE":    round(float(fit["metrics_full"]["MAE"]), 4),
            "RMSE_typical":        round(float(fit["metrics_typical"]["RMSE"]), 4),
            "RMSE_full":           round(float(fit["metrics_full"]["RMSE"]), 4),
            "MdAE_typical":        round(float(fit["metrics_typical"]["MdAE"]), 4),
            "MdAE_full":           round(float(fit["metrics_full"]["MdAE"]), 4),
            "n_params":            int(fit["n_params"]),
            "epochs_used":         int(fit["epochs_used"]),
            "best_params":         _json.dumps(bp, ensure_ascii=False),
            "time_s":              round(time.time() - t0, 1),
        })
        print(f"  {arch_name:<16s} | val={study.best_value:.3f} | typ={fit['metrics_typical']['MAE']:.3f} | full={fit['metrics_full']['MAE']:.3f} | t={time.time()-t0:.0f}s")
        _save_cached_rows(rows, cache_path)
        clear_device_cache()

    out = pd.DataFrame(rows).sort_values(
        ["cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE"]
    ).reset_index(drop=True)
    _save_cached_rows(out.to_dict("records"), cache_path)
    display(out)
    return out


def run_variant_dl_baseline(Xtr_aug, Xte_typ_aug, Xte_full_aug, title="baseline", archs=None, cache_path=None):
    if archs is None:
        archs = DL_RUN_ARCHS

    print("=" * 95)
    print(title)
    print("=" * 95)

    setup_dl_for_variant(
        Xtr_aug, meta_ytr0,
        Xte_typ_aug, meta_yte_typ0,
        Xte_full_aug, meta_yte0,
    )

    rows = _load_cached_rows(cache_path)
    done_models = {str(r.get("model")) for r in rows}
    for arch_name in archs:
        if arch_name in done_models:
            print(f"  [SKIP] {arch_name}: already cached in {cache_path}")
            continue
        if arch_name == "TabPFN" and not (RUN_TABPFN and HAS_TABPFN):
            continue
        if arch_name != "GrowNet" and arch_name != "TabPFN" and arch_name not in DEFAULTS:
            print(f"  skip {arch_name}: no default builder")
            continue

        print(f"\n{'-' * 72}\n  BASELINE DEFAULT | {arch_name}\n{'-' * 72}")
        t0 = time.time()
        seed_everything(SEED)

        try:
            if arch_name == "GrowNet":
                stages, val_mae, total_ep = train_grownet(
                    n_stages=5, hidden=32, lr=0.01, wd=1e-4,
                    step_size=0.1, bs=256, patience=30, dropout=0.1,
                )
                n_params = int(sum(sum(p.numel() for p in s.parameters()) for s in stages))
                m_typ = eval_grownet_on_typical_holdout(stages, step_size=0.1)
                m_full = eval_grownet_on_full_holdout(stages, step_size=0.1)
                epochs_used = int(total_ep)
            elif arch_name == "TabPFN":
                pfn_device = "cuda" if torch.cuda.is_available() else "cpu"
                tabpfn = TabPFNRegressor(device=pfn_device, n_estimators=8)
                X_pfn_tr = X_dl_tr.copy()
                y_pfn_tr = y_dl_tr.copy()
                if len(X_pfn_tr) > 3000:
                    idx = np.random.RandomState(SEED).choice(len(X_pfn_tr), 3000, replace=False)
                    X_pfn_tr = X_pfn_tr[idx]
                    y_pfn_tr = y_pfn_tr[idx]
                tabpfn.fit(X_pfn_tr, y_pfn_tr)
                val_pred = tabpfn.predict(X_dl_va)
                val_mae = mean_absolute_error(y_dl_va, val_pred)
                m_typ, m_full = eval_tabpfn_holdouts(tabpfn_model=tabpfn, use_full_train_scaler=False)
                n_params = 0
                epochs_used = 0
            else:
                model = DEFAULTS[arch_name]()
                aux_w = AUX_W.get(arch_name, 0.0)
                n_params = int(sum(p.numel() for p in model.parameters()))
                model, val_mae, epochs_used = train_model(
                    model,
                    epochs=300,
                    lr=1e-3,
                    wd=1e-4,
                    bs=256,
                    patience=30,
                    aux_w=aux_w,
                )
                if arch_name == "TabR":
                    model.build_store(X_tr_t.to(device), y_tr_t.to(device))
                m_typ = eval_on_typical_holdout(model)
                m_full = eval_on_full_holdout(model)
        except Exception as e:
            print(f"  {arch_name}: baseline failed: {type(e).__name__}: {e}")
            _save_cached_rows(rows, cache_path)
            clear_device_cache()
            continue

        elapsed = time.time() - t0
        rows.append({
            "model": arch_name,
            "cv_MAE_internal": round(float(val_mae), 4),
            "refit_val_MAE": round(float(val_mae), 4),
            "holdout_typical_MAE": round(float(m_typ["MAE"]), 4),
            "holdout_full_MAE": round(float(m_full["MAE"]), 4),
            "RMSE_typical": round(float(m_typ["RMSE"]), 4),
            "RMSE_full": round(float(m_full["RMSE"]), 4),
            "MdAE_typical": round(float(m_typ["MdAE"]), 4),
            "MdAE_full": round(float(m_full["MdAE"]), 4),
            "n_params": int(n_params),
            "epochs_used": int(epochs_used),
            "best_params": _json.dumps(serialize_params(BASELINE_TRIAL_PARAMS.get(arch_name, {})), ensure_ascii=False),
            "time_s": round(float(elapsed), 1),
        })
        print(f"  {arch_name:<16s} | val={float(val_mae):.3f} | typ={m_typ['MAE']:.3f} | full={m_full['MAE']:.3f} | t={elapsed:.0f}s")
        _save_cached_rows(rows, cache_path)
        clear_device_cache()

    out = pd.DataFrame(rows).sort_values(
        ["cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE"]
    ).reset_index(drop=True)
    _save_cached_rows(out.to_dict("records"), cache_path)
    display(out)
    return out


In [ ]:
print("Core DL architectures from DL_baseline.ipynb:")
print(", ".join(DEFAULTS.keys()))
print("Extra models: GrowNet; TabPFN if installed and RUN_TABPFN=1")
print("Feature spaces:")
for name in ["llm_only"] + [f"cluster_before_llm__{t}" for t in cluster_before_llm_variants] + [f"llm_then_cluster__{t}" for t in llm_then_cluster_variants]:
    print("-", name)
print("Outputs are written under:", DL_ARTIFACT_DIR)


In [ ]:
DL_ARTIFACT_DIR = ARTIFACT_DIR / "dl_tuning"
DL_BASELINE_ARTIFACT_DIR = ARTIFACT_DIR / "dl_baseline_default_params"
DL_SELECTED_TUNING_DIR = DL_ARTIFACT_DIR / "selected_scheme_tuning"
DL_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
DL_BASELINE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
DL_SELECTED_TUNING_DIR.mkdir(parents=True, exist_ok=True)

existing_compare_path = DL_ARTIFACT_DIR / "dl_llm_cluster_compare_results.csv"
if RUN_EXISTING_RESULTS_ONLY and existing_compare_path.exists():
    print("RUN_EXISTING_RESULTS_ONLY=1; loading saved DL results from", existing_compare_path)
    dl_compare_df = pd.read_csv(existing_compare_path)
    display(dl_compare_df.sort_values(["holdout_typical_MAE", "holdout_full_MAE"]).head(30))
else:
    def add_experiment_name(df_res, name):
        tmp = df_res.copy()
        tmp["experiment"] = name
        return tmp

    feature_space_bundles = {
        "llm_only": {
            "title_prefix": "DL VARIANT 1",
            "Xtr": Xtr_llm_only,
            "Xte_typ": Xte_llm_only_typ,
            "Xte_full": Xte_llm_only_full,
        }
    }
    for tag, bundle in cluster_before_llm_variants.items():
        feature_space_bundles[f"cluster_before_llm__{tag}"] = {
            "title_prefix": "DL VARIANT 2",
            "Xtr": bundle["Xtr"],
            "Xte_typ": bundle["Xte_typ"],
            "Xte_full": bundle["Xte_full"],
        }
    for tag, bundle in llm_then_cluster_variants.items():
        feature_space_bundles[f"llm_then_cluster__{tag}"] = {
            "title_prefix": "DL VARIANT 3",
            "Xtr": bundle["Xtr"],
            "Xte_typ": bundle["Xte_typ"],
            "Xte_full": bundle["Xte_full"],
        }

    dl_baseline_result_frames = []

    # 1) Baseline/default DL по всем схемам. Это дешево относительно Optuna и нужно,
    #    чтобы выбрать одну лучшую feature-схему для каждой DL-архитектуры.
    for exp_name, bundle in feature_space_bundles.items():
        baseline_res = run_variant_dl_baseline(
            bundle["Xtr"], bundle["Xte_typ"], bundle["Xte_full"],
            title=f"DL BASELINE DEFAULT: {exp_name}",
            cache_path=DL_BASELINE_ARTIFACT_DIR / f"{exp_name}_dl_baseline_results.csv",
        )
        baseline_res.to_csv(DL_BASELINE_ARTIFACT_DIR / f"{exp_name}_dl_baseline_results.csv", index=False)
        dl_baseline_result_frames.append(add_experiment_name(baseline_res, exp_name))

    dl_baseline_compare_df = pd.concat(dl_baseline_result_frames, ignore_index=True)
    dl_baseline_best_by_exp_cv = (
        dl_baseline_compare_df.sort_values(["cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE"])
        .groupby("experiment", as_index=False).head(1).reset_index(drop=True)
    )
    dl_baseline_best_by_exp_holdout = (
        dl_baseline_compare_df.sort_values(["holdout_typical_MAE", "holdout_full_MAE", "cv_MAE_internal"])
        .groupby("experiment", as_index=False).head(1).reset_index(drop=True)
    )
    dl_baseline_global_holdout_report = dl_baseline_compare_df.sort_values(
        ["holdout_typical_MAE", "holdout_full_MAE", "cv_MAE_internal"]
    ).reset_index(drop=True)
    dl_best_scheme_by_model = (
        dl_baseline_compare_df.sort_values(["model", "cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE"])
        .groupby("model", as_index=False).head(1).reset_index(drop=True)
    )

    print("DL baseline/default: best per experiment by inner val MAE:")
    display(dl_baseline_best_by_exp_cv)
    print("DL baseline/default: best feature scheme for each DL model by inner val MAE:")
    display(dl_best_scheme_by_model[["model", "experiment", "cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE"]])
    print("DL baseline/default: global top-20 by holdout_typical_MAE:")
    display(dl_baseline_global_holdout_report.head(20))

    dl_baseline_compare_df.to_csv(DL_BASELINE_ARTIFACT_DIR / "dl_llm_cluster_baseline_compare_results.csv", index=False)
    dl_baseline_best_by_exp_cv.to_csv(DL_BASELINE_ARTIFACT_DIR / "dl_llm_cluster_baseline_best_by_experiment_cv.csv", index=False)
    dl_baseline_best_by_exp_holdout.to_csv(DL_BASELINE_ARTIFACT_DIR / "dl_llm_cluster_baseline_best_by_experiment_holdout.csv", index=False)
    dl_baseline_global_holdout_report.to_csv(DL_BASELINE_ARTIFACT_DIR / "dl_llm_cluster_baseline_global_holdout_report.csv", index=False)
    dl_best_scheme_by_model.to_csv(DL_BASELINE_ARTIFACT_DIR / "dl_best_feature_scheme_by_model_cv.csv", index=False)

    # 2) Optuna-тюнинг только один раз на модель: каждая DL-архитектура тюнится
    #    только на своей лучшей схеме из baseline/default шага.
    selected_archs_by_experiment = {
        exp_name: sorted(dl_best_scheme_by_model.loc[dl_best_scheme_by_model["experiment"] == exp_name, "model"].astype(str).tolist())
        for exp_name in feature_space_bundles
    }
    selected_archs_by_experiment = {k: v for k, v in selected_archs_by_experiment.items() if v}
    print("Selected architectures for one-pass tuning by feature scheme:")
    for exp_name, archs in selected_archs_by_experiment.items():
        print(f"  {exp_name}: {archs}")

    dl_all_result_frames = []
    for exp_name, archs in selected_archs_by_experiment.items():
        bundle = feature_space_bundles[exp_name]
        res = run_variant_dl_tuning(
            bundle["Xtr"], bundle["Xte_typ"], bundle["Xte_full"],
            title=f"DL TUNED SELECTED SCHEME: {exp_name}",
            archs=archs,
            cache_path=DL_SELECTED_TUNING_DIR / f"{exp_name}_dl_tuned_results.csv",
            study_dir=DL_SELECTED_TUNING_DIR / "optuna_sqlite",
        )
        res = res[res["model"].astype(str).isin(archs)].reset_index(drop=True)
        res.to_csv(DL_SELECTED_TUNING_DIR / f"{exp_name}_dl_tuned_results.csv", index=False)
        dl_all_result_frames.append(add_experiment_name(res, exp_name))

    dl_compare_df = pd.concat(dl_all_result_frames, ignore_index=True)
    dl_best_by_exp_cv = (
        dl_compare_df.sort_values(["cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE"])
        .groupby("experiment", as_index=False).head(1).reset_index(drop=True)
    )
    dl_best_by_exp_holdout = (
        dl_compare_df.sort_values(["holdout_typical_MAE", "holdout_full_MAE", "cv_MAE_internal"])
        .groupby("experiment", as_index=False).head(1).reset_index(drop=True)
    )
    dl_tuned_best_by_model = (
        dl_compare_df.sort_values(["model", "cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE"])
        .groupby("model", as_index=False).head(1).reset_index(drop=True)
    )
    dl_global_holdout_report = dl_compare_df.sort_values(
        ["holdout_typical_MAE", "holdout_full_MAE", "cv_MAE_internal"]
    ).reset_index(drop=True)

    print("DL tuned selected schemes: best per experiment by inner val MAE:")
    display(dl_best_by_exp_cv)
    print("DL tuned selected schemes: one tuned result per DL model:")
    display(dl_tuned_best_by_model[["model", "experiment", "cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE"]])
    print("DL tuned selected schemes: global top-20 by holdout_typical_MAE:")
    display(dl_global_holdout_report.head(20))

    dl_compare_df.to_csv(DL_ARTIFACT_DIR / "dl_llm_cluster_compare_results.csv", index=False)
    dl_best_by_exp_cv.to_csv(DL_ARTIFACT_DIR / "dl_llm_cluster_best_by_experiment_cv.csv", index=False)
    dl_best_by_exp_holdout.to_csv(DL_ARTIFACT_DIR / "dl_llm_cluster_best_by_experiment_holdout.csv", index=False)
    dl_tuned_best_by_model.to_csv(DL_ARTIFACT_DIR / "dl_tuned_best_by_model_selected_scheme_cv.csv", index=False)
    dl_global_holdout_report.to_csv(DL_ARTIFACT_DIR / "dl_llm_cluster_global_holdout_report.csv", index=False)
    raw_cluster_summary_df.to_csv(ARTIFACT_DIR / "raw_cluster_config_summary.csv", index=False)
    cluster_after_summary_df.to_csv(ARTIFACT_DIR / "cluster_after_llm_config_summary.csv", index=False)

    (DL_ARTIFACT_DIR / "dl_llm_experiment_summary.json").write_text(
        json.dumps({
            "selection_rule": "First run DL default/baseline for every feature scheme; for each DL model select the feature scheme with the lowest inner val MAE; tune each DL model once on that selected scheme only.",
            "baseline_source": "DL_baseline.ipynb default parameters (24 core architectures + GrowNet + optional TabPFN)",
            "tuning_source": "DL_tuning.ipynb search space/trial policy (ARCH_TRIALS=50 for each selected architecture)",
            "provider": PROVIDER,
            "backend_tag": BACKEND_TAG,
            "trials_per_arch": ARCH_TRIALS,
            "baseline_artifact_dir": str(DL_BASELINE_ARTIFACT_DIR),
            "selected_tuning_artifact_dir": str(DL_SELECTED_TUNING_DIR),
            "feature_spaces": list(feature_space_bundles.keys()),
            "selected_archs_by_experiment": selected_archs_by_experiment,
        }, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print("Saved DL comparative results to", DL_ARTIFACT_DIR)
